# Building the deck tables

Turns the decklist text files in `data/external/decks/` into three tables in `data/processed/`. It needs `cards.parquet` and `moves.parquet` to exist already.

| Table | One row per | Main columns |
|---|---|---|
| `decks` | deck | `deck_id`, `mode`, `n_cards`, `n_distinct`, `n_lines`, `unresolved_weight`, `source` |
| `deck_cards` | card in a deck | `deck_id`, `card_id`, `copies`, `avg_count` |
| `deck_profile` | card seen in a deck's list | `deck_id`, `card_id`, `avg_count` |

`deck_id` is the filename without `.txt`. To add a deck, save its list as a new `.txt` file in the folder and re-run the notebook.

In [56]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from tcg_bot.paths import ROOT
PROC = ROOT / "data" / "processed"
DECK_DIR = ROOT / "data" / "external" / "decks"

cards = pd.read_parquet(PROC / "cards.parquet")
moves = pd.read_parquet(PROC / "moves.parquet")

# deck rules come from the rules table (run plays.ipynb first), not from numbers typed here
_c = pd.read_parquet(PROC / "game_constants.parquet").set_index("key")
USED = ["deck_size", "max_copies_per_name", "ace_spec_max_per_deck", "min_basic_pokemon_in_deck"]
assert (_c.loc[USED, "status"] == "verified").all(), _c.loc[USED, ["value", "status"]]
DECK_SIZE = int(_c.at["deck_size", "value"])
MAX_COPIES = int(_c.at["max_copies_per_name", "value"])
ACE_SPEC_MAX = int(_c.at["ace_spec_max_per_deck", "value"])
MIN_BASICS = int(_c.at["min_basic_pokemon_in_deck", "value"])

## 1. Parsing a deck file

A deck file has one card per line: a count, the card name, and optionally the set code and card number.

```
# source: limitlesstcg.com, 2026-11-23     <- '#' starts a comment
Pokémon: 16
3 Riolu MEG 76
4 Ultra Ball
```

- Section headers (`Pokémon: 16` or `Pokémon (16.47)`) only mark the section. The numbers in them are ignored.
- Blank lines and a `Total Cards:` footer are skipped.
- Set and number are optional. They are only needed for cards that have several versions (for example Riolu or Shaymin).
- A line that can't be read raises an error with its line number.

In [57]:
def parse_limitless(text):
    section, rows = None, []
    for n, line in enumerate(text.splitlines(), 1):
        line = line.split("#")[0].strip()                       # '#' starts a comment
        if not line or line.lower().startswith("total cards"):  # blank lines and the "Total Cards: 60" footer
            continue
        m = re.match(r"^(Pokémon|Trainer|Energy)\s*[:(]\s*[\d.]+\)?$", line)   # section header
        if m:
            section = m.group(1)
            continue
        m = re.match(r"^(\d+(?:\.\d+)?)\s+(.+?)(?:\s+([A-Z0-9][A-Z0-9-]{1,5})\s+(\d+))?$", line)
        if not m:
            raise ValueError(f"line {n}: can't read {line!r}")
        rows.append({"section": section, "avg_count": float(m[1]), "name": m[2],
                     "set": m[3], "number": int(m[4]) if m[4] else None})
    return pd.DataFrame(rows)

## 2. Resolving lines to cards

Every line has to become a `card_id` from the `cards` table. `resolve` tries, in order:

1. **`print_alias`**: an exact `(name, set, number)` entry in `PRINT_ALIASES`.
2. **`name`**: the dataset has exactly one card with that name (after applying `ALIASES`).
3. **`set+number`**: the name has several versions, so the set and number pick one.

Anything else is `ambiguous` (several versions and no match) or `unmatched` (no such card).

- `ALIASES` maps names Limitless writes differently from our dataset, for example `Fighting Energy` to `Basic {F} Energy`.
- `PRINT_ALIASES` maps a specific printing our dataset doesn't list to a card. **Each entry is a hand-made assumption. Check it against the real card.** A wrong entry silently puts the wrong card in the deck.

In [58]:
# Limitless name -> name used in our dataset
ALIASES = {
    "Fighting Energy": "Basic {F} Energy",
    "Rocky Fighting Energy": "Rock Fighting Energy",
}
# (lowercase name, set, number) -> card_id, for prints our dataset doesn't list. CHECK EACH ONE BY HAND.
PRINT_ALIASES = {
    ("snorunt", "ASC", 227): 860,        # assumed alternate print of Snorunt ASC 46 (70 HP, Chilly)
    ("chien-pao", "PR-SV", 152): 209,    # assumed Chien-Pao SSP 56 (Water, Snow Sink); POR 54 is Darkness
}

def resolve(raw, cards):
    ids_by_name = cards.groupby(cards["card_name"].str.lower())["card_id"].agg(list)
    out_id, how = [], []
    for r in raw.itertuples():
        key = (r.name.lower(), r.set, r.number)
        if key in PRINT_ALIASES:
            out_id.append(PRINT_ALIASES[key]); how.append("print_alias")
            continue
        ids = ids_by_name.get(ALIASES.get(r.name, r.name).lower(), [])
        if len(ids) == 1:                       # one card with that name
            out_id.append(ids[0]); how.append("name")
        elif len(ids) > 1:                      # several versions: use set + number
            hit = cards[cards["card_id"].isin(ids)
                        & (cards["expansion"] == r.set) & (cards["collection_no"] == r.number)]
            out_id.append(hit["card_id"].iloc[0] if len(hit) == 1 else pd.NA)
            how.append("set+number" if len(hit) == 1 else "ambiguous")
        else:
            out_id.append(pd.NA); how.append("unmatched")
    return raw.assign(card_id=pd.array(out_id, dtype="Int64"), matched_by=how)

## 3. From lines to a 60-card deck

`build_deck` first adds up lines that point to the same card (different prints), then works in one of two modes:

- **exact**: all counts are whole numbers. The list is used exactly as written. Nothing is added or dropped, and an unresolved line or a total other than 60 stops the build with an error.
- **average**: counts have decimals, as on a Limitless average list. The counts are rounded down, then the leftover copies go to the largest fractions. At most 4 copies of a card (Basic Energy is exempt) and at most one ACE SPEC are allowed. Up to 0.5 cards of unresolved weight are tolerated.

The mode is stored in the `decks` table.

In [59]:
def build_deck(resolved, cards, total=DECK_SIZE, max_copies=MAX_COPIES, tolerance=0.5):
    """Whole-number list: used exactly as written (never padded or guessed).
    Average list (fractional counts): rounded to a legal 60."""
    missing = resolved.loc[resolved["card_id"].isna()]
    df = resolved[resolved["card_id"].notna()].copy()
    df["card_id"] = df["card_id"].astype(int)
    df = df.groupby("card_id", as_index=False).agg(avg_count=("avg_count", "sum"), line_order=("line_order", "min"))   # same card on several lines (prints)
    df = df.merge(cards[["card_id", "card_name", "family", "card_kind", "is_ace_spec"]], on="card_id")
    is_average = bool((resolved["avg_count"] % 1 != 0).any())

    if not is_average:
        if len(missing):
            raise ValueError("unresolved lines in a whole-number list:\n"
                             + missing[["avg_count", "name", "set", "number", "matched_by"]].to_string(index=False))
        df["copies"] = df["avg_count"].astype(int)
        if df["copies"].sum() != total:
            raise ValueError(f"deck has {df['copies'].sum()} cards, expected {total}")
        return df, "exact"

    if missing["avg_count"].sum() > tolerance:
        raise ValueError("too much of the average list is unresolved:\n"
                         + missing[["avg_count", "name", "set", "number", "matched_by"]].to_string(index=False))
    cap = np.where(df["card_kind"] == "Basic Energy", 10**6, max_copies)
    df["copies"] = np.minimum(np.floor(df["avg_count"]).astype(int), cap)
    need = total - df["copies"].sum()
    ace_in = int(df.loc[df["is_ace_spec"], "copies"].sum())
    frac = df["avg_count"] - np.floor(df["avg_count"])
    for i in frac.sort_values(ascending=False).index:        # extra copies go to the biggest fractions
        if need == 0 or frac[i] <= 0:
            break
        if df.at[i, "copies"] >= cap[df.index.get_loc(i)]:
            continue
        if df.at[i, "is_ace_spec"] and ace_in >= ACE_SPEC_MAX:
            continue
        df.at[i, "copies"] += 1
        need -= 1
        ace_in += int(df.at[i, "is_ace_spec"])
    if need != 0:
        raise ValueError(f"could not reach {total} cards, still need {need}")
    return df[df["copies"] > 0].reset_index(drop=True), "average"

## 4. Deck rules

`validate_deck` checks the final list:

- exactly 60 cards;
- at most 4 copies of a card by name (Basic Energy is exempt);
- at most 1 ACE SPEC card;
- at least 1 Basic Pokémon.

These are the rules from the pipeline plan. They still need to be checked against the official rulebook.

In [60]:
def validate_deck(deck, total=DECK_SIZE, max_copies=MAX_COPIES):
    by_name = deck.groupby("card_name").agg(
        copies=("copies", "sum"),
        basic_energy=("card_kind", lambda s: (s == "Basic Energy").all()))
    assert deck["copies"].sum() == total, deck["copies"].sum()
    over = by_name[(~by_name["basic_energy"]) & (by_name["copies"] > max_copies)]
    assert over.empty, f"more than {max_copies} copies:\n{over}"
    ace = deck[deck["is_ace_spec"]]
    assert ace["copies"].sum() <= ACE_SPEC_MAX, f"more than {ACE_SPEC_MAX} ACE SPEC card(s):\n{ace[['card_name', 'copies']]}"
    assert deck.loc[deck["card_kind"] == "Basic Pokémon", "copies"].sum() >= MIN_BASICS, "not enough Basic Pokémon in the deck"

## 5. Building every deck file

Loops over the `.txt` files in the folder, then parses, resolves, builds and validates each deck.

- It prints one row per deck with how its lines were matched (`name`, `set+number`, `print_alias`, `unmatched`...).
- If a deck fails, the reason is printed under `FAILED`, and a `RuntimeError` stops the notebook before anything is saved.
- `unresolved_weight` in `decks` is the number of cards that couldn't be matched. It should be 0 for an exact list.

In [61]:
decks, deck_cards, deck_profile, errors, reports = [], [], [], {}, []
for path in sorted(DECK_DIR.glob("*.txt")):
    deck_id = path.stem
    try:
        raw = parse_limitless(path.read_text(encoding="utf-8"))
        resolved = resolve(raw, cards)
        resolved["line_order"] = range(1, len(resolved) + 1)
        deck, mode = build_deck(resolved, cards)
        validate_deck(deck)
    except (ValueError, AssertionError) as e:
        errors[deck_id] = str(e)
        continue
    bad = resolved[resolved["card_id"].isna()]
    reports.append({"deck_id": deck_id, "lines": len(raw), **resolved["matched_by"].value_counts().to_dict()})
    profile = (resolved[resolved["card_id"].notna()].assign(card_id=lambda d: d["card_id"].astype(int))
               .groupby("card_id", as_index=False)["avg_count"].sum().assign(deck_id=deck_id))
    deck_profile.append(profile[["deck_id", "card_id", "avg_count"]])
    deck_cards.append(deck.assign(deck_id=deck_id)[["deck_id", "card_id", "copies", "avg_count", "line_order"]])
    decks.append({"deck_id": deck_id, "mode": mode, "n_cards": int(deck["copies"].sum()),
                  "n_distinct": len(deck), "n_lines": len(raw),
                  "unresolved_weight": round(float(bad["avg_count"].sum()), 2),
                  "source": "limitless_average" if mode == "average" else "decklist"})

print(pd.DataFrame(reports).fillna(0).astype({c: int for c in pd.DataFrame(reports).columns if c != "deck_id"}, errors="ignore"))
for deck_id, msg in errors.items():
    print(f"\nFAILED  {deck_id}\n{msg}")
if errors:
    raise RuntimeError(f"fix {list(errors)} before saving")

decks = pd.DataFrame(decks)
deck_cards = pd.concat(deck_cards, ignore_index=True)
deck_profile = pd.concat(deck_profile, ignore_index=True)
decks

        deck_id  lines  name  set+number  print_alias
0       lopunny     31    25           4            2
1  mega_lucario     22    21           1            0


,deck_id,mode,n_cards,n_distinct,n_lines,unresolved_weight,source
0,lopunny,exact,60,26,31,0.0,decklist
1,mega_lucario,exact,60,22,22,0.0,decklist


## 6. Reading the decks back

Joins `deck_cards` to `cards` to print each deck by card name, sorted by card kind. Use this to check the lists against the original files.

In [62]:
view = deck_cards.merge(cards[["card_id", "card_name", "card_kind", "expansion", "collection_no"]], on="card_id")
for deck_id, g in view.groupby("deck_id"):
    print(f"\n== {deck_id}  ({g['copies'].sum()} cards)")
    print(g.sort_values(["card_kind", "copies"], ascending=[True, False])
           [["copies", "card_name", "card_kind", "expansion", "collection_no"]].to_string(index=False))


== lopunny  (60 cards)
 copies              card_name       card_kind expansion  collection_no
      3       Basic {W} Energy    Basic Energy       SVE              3
      2              Dunsparce   Basic Pokémon       JTG            120
      2                Buneary   Basic Pokémon       PFL             83
      2                Snorunt   Basic Pokémon       ASC             46
      1              Dunsparce   Basic Pokémon       TEF            128
      1              Fan Rotom   Basic Pokémon       SCR            118
      1              Chien-Pao   Basic Pokémon       SSP             56
      1                Psyduck   Basic Pokémon       ASC             39
      4     Buddy-Buddy Poffin            Item       TEF            144
      4               Poké Pad            Item       POR             81
      3             Ultra Ball            Item       SVI            196
      2        Night Stretcher            Item       SFA             61
      2           Pokégear 3.0          

## 7. Scope and saving

- `in_scope` counts the rows of `moves` whose card appears in at least one deck. This is the amount of effect text the engine has to support first.
- The three tables are saved to `data/processed/`: `decks.parquet`, `deck_cards.parquet` and `deck_profile.parquet`.

`deck_cards` is the playable list. `deck_profile` keeps the unrounded `avg_count` for each card, and the opponent-inference step will use it later.

In [63]:
in_scope = moves["card_id"].isin(deck_cards["card_id"])
print(in_scope.sum(), "of", len(moves), "moves are in at least one playable deck")

decks.to_parquet(PROC / "decks.parquet", index=False)
deck_cards.to_parquet(PROC / "deck_cards.parquet", index=False)
deck_profile.to_parquet(PROC / "deck_profile.parquet", index=False)

54 of 2265 moves are in at least one playable deck


## 8. Energy payment table

Builds `energy_provides`: what each of the 25 Energy cards can pay when a Pokémon attacks. Basic Energy comes from `cards.type`. The 17 Special Energy cards are hand-labelled in `configs/energy_provides.csv` from their card text. The last check confirms every Energy card used in a deck has a row.

Needs `deck_cards.parquet`, so run the deck cells above first.

In [64]:
import pandas as pd
from pathlib import Path

from tcg_bot.paths import ROOT
CONFIGS = ROOT / "configs"
PROC = ROOT / "data" / "processed"

cards = pd.read_parquet(PROC / "cards.parquet")
deck_cards = pd.read_parquet(PROC / "deck_cards.parquet")
special = pd.read_csv(CONFIGS / "energy_provides.csv", encoding="utf-8")

In [65]:
basic = cards[cards["card_kind"] == "Basic Energy"]
energy_basic = pd.DataFrame({
    "card_id": basic["card_id"].values,
    "card_name": basic["card_name"].values,
    "provides": basic["type"].str.strip("{}").values,
    "amount": 1,
    "condition": pd.NA, "cond_provides": pd.NA, "cond_amount": pd.NA,
    "attach_restriction": pd.NA, "discard_end_of_turn": False, "status": "verified",
})
energy_provides = pd.concat([energy_basic, special], ignore_index=True)
for c in ["condition", "cond_provides", "attach_restriction"]:
    energy_provides[c] = energy_provides[c].astype("string")
energy_provides["amount"] = energy_provides["amount"].astype(int)
energy_provides["cond_amount"] = energy_provides["cond_amount"].astype("Int64")
energy_provides["discard_end_of_turn"] = energy_provides["discard_end_of_turn"].astype(bool)

In [66]:
TYPES = {"C", "G", "R", "W", "L", "P", "F", "D", "M", "N"}
PROVIDES = TYPES | {"ALL", "P|D"}
CONDITIONS = {"attached_to_stage2", "attached_to_basic", "attached_to_evolution"}
RESTRICTIONS = {"team_rocket"}

energy_ids = set(cards.loc[cards["family"] == "energy", "card_id"])
assert set(energy_provides["card_id"]) == energy_ids, (
    f"missing: {energy_ids - set(energy_provides['card_id'])} extra: {set(energy_provides['card_id']) - energy_ids}")
assert energy_provides["card_id"].is_unique, f"duplicate card_id: {energy_provides.loc[energy_provides['card_id'].duplicated(), 'card_id'].tolist()}"
assert set(energy_provides["provides"]) <= PROVIDES, set(energy_provides["provides"]) - PROVIDES
assert (energy_provides["amount"] >= 1).all(), "amount must be at least 1"

has_cond = energy_provides["condition"].notna()
assert set(energy_provides.loc[has_cond, "condition"]) <= CONDITIONS, f"unknown condition: {set(energy_provides.loc[has_cond, 'condition']) - CONDITIONS}"
assert energy_provides.loc[has_cond, ["cond_provides", "cond_amount"]].notna().all().all(), "a condition needs cond_provides and cond_amount"
assert energy_provides.loc[~has_cond, ["cond_provides", "cond_amount"]].isna().all().all(), "cond_* set without a condition"
assert set(energy_provides.loc[has_cond, "cond_provides"]) <= PROVIDES, "unknown cond_provides"
assert set(energy_provides["attach_restriction"].dropna()) <= RESTRICTIONS, "unknown attach_restriction"

# names in the CSV must match the cards table (catches a wrong card_id)
names = cards.set_index("card_id")["card_name"]
wrong = energy_provides[energy_provides["card_name"] != energy_provides["card_id"].map(names)]
assert wrong.empty, wrong[["card_id", "card_name"]]

# every Energy card used in any deck must be covered
used = deck_cards.loc[deck_cards["card_id"].isin(energy_ids), "card_id"]
assert used.isin(energy_provides["card_id"]).all(), "an Energy card used in a deck has no row"

In [67]:
view = (deck_cards[deck_cards["card_id"].isin(energy_ids)]
        .merge(energy_provides[["card_id", "card_name", "provides", "amount", "condition"]], on="card_id"))
print(view[["deck_id", "copies", "card_name", "provides", "amount", "condition"]].to_string(index=False))
energy_provides.to_parquet(PROC / "energy_provides.parquet", index=False)
print(energy_provides.shape)

     deck_id  copies        card_name provides  amount condition
     lopunny       3 Basic {W} Energy        W       1      <NA>
     lopunny       4      Mist Energy        C       1      <NA>
     lopunny       1 Enriching Energy        C       1      <NA>
mega_lucario      11 Basic {F} Energy        F       1      <NA>
(25, 10)


check

In [68]:
# ---- manual check of the card tags (expects the deck notebook's earlier cells to have run) ----
CHECK_DECK = "lopunny"          # change to "mega_lucario" (or any deck file name) to check another deck

try:
    from tcg_bot.decks import expand_deck
    _source = "tcg_bot.decks"
except ModuleNotFoundError:     # src/tcg_bot/decks.py not created yet: use a copy so the check still runs
    from dataclasses import dataclass

    @dataclass(frozen=True)
    class CardInstance:
        uid: int
        tag: str
        card_id: int

    def expand_deck(deck_cards, cards, deck_id):
        rows = deck_cards[deck_cards["deck_id"] == deck_id].sort_values("line_order")
        if rows.empty:
            raise ValueError(f"unknown deck_id {deck_id!r}")
        family = cards.set_index("card_id")["family"]
        counters, out = {}, []
        for row in rows.itertuples():
            for _ in range(int(row.copies)):
                fam = family[int(row.card_id)]
                counters[fam] = counters.get(fam, 0) + 1
                out.append(CardInstance(uid=len(out), tag=f"{deck_id}_{fam}_{counters[fam]}", card_id=int(row.card_id)))
        return out
    _source = "a copy defined in this cell (tcg_bot/decks.py does not exist yet)"
print("expand_deck comes from:", _source)

names = cards.set_index("card_id")["card_name"]
family = cards.set_index("card_id")["family"]

# the .txt lines, numbered exactly the way line_order counts them
raw = parse_limitless((DECK_DIR / f"{CHECK_DECK}.txt").read_text(encoding="utf-8"))
lines = resolve(raw, cards)
lines["line_order"] = range(1, len(lines) + 1)
lines["card_name"] = lines["card_id"].map(names)

# the 60 tagged cards
table = pd.DataFrame(expand_deck(deck_cards, cards, CHECK_DECK))
table["card_name"] = table["card_id"].map(names)
first_line = (lines.dropna(subset=["card_id"]).assign(card_id=lambda d: d["card_id"].astype(int))
              .groupby("card_id")["line_order"].min())
table["from_line"] = table["card_id"].map(first_line)

print(f"\n== {CHECK_DECK}")
print("\n1) the .txt lines (line number = line_order)")
print(lines[["line_order", "section", "avg_count", "name", "set", "number", "card_name", "matched_by"]].to_string(index=False))
print("\n2) the tagged cards (from_line = the .txt line the card first appears on)")
print(table[["uid", "tag", "card_id", "card_name", "from_line"]].to_string(index=False))

# 3) automatic checks, each printed as OK or with what is wrong
results = []
def check(label, ok, detail=""):
    results.append(ok)
    print(f"  {'OK ' if ok else 'BAD'} {label}" + (f"  -> {detail}" if not ok and detail else ""))

print("\n3) checks")
check("60 cards", len(table) == 60, f"{len(table)} cards")
check("uid runs 0..59", table["uid"].tolist() == list(range(len(table))))
check("every tag is unique", table["tag"].is_unique, str(table.loc[table["tag"].duplicated(), "tag"].tolist()))
for fam in ("pokemon", "trainer", "energy"):
    nums = [int(t.rsplit("_", 1)[1]) for t in table["tag"] if t.startswith(f"{CHECK_DECK}_{fam}_")]
    check(f"{fam} tags numbered 1..{len(nums)} with no gaps", nums == list(range(1, len(nums) + 1)), str(nums))
check("the family inside each tag matches the card", all(t.split("_")[-2] == family[c] for t, c in zip(table["tag"], table["card_id"])))
want = (lines.dropna(subset=["card_id"]).assign(card_id=lambda d: d["card_id"].astype(int))
        .groupby("card_id")["avg_count"].sum().astype(int))
have = table.groupby("card_id").size()
check("copies of each card match the .txt counts", want.sort_index().to_dict() == have.sort_index().to_dict(),
      str({k: (want.get(k), have.get(k)) for k in set(want.index) | set(have.index) if want.get(k) != have.get(k)}))
check("tags follow the .txt order (from_line never goes backwards)", table["from_line"].is_monotonic_increasing)
sec = lines.dropna(subset=["card_id"]).assign(fam=lambda d: d["card_id"].astype(int).map(family),
                                              sec=lambda d: d["section"].str.lower().str.replace("é", "e"))
check("the .txt section agrees with each card's family", (sec["sec"] == sec["fam"]).all(),
      str(sec.loc[sec["sec"] != sec["fam"], ["name", "section", "fam"]].values.tolist()))
check("no .txt line is unresolved", lines["card_id"].notna().all(), str(lines.loc[lines["card_id"].isna(), "name"].tolist()))
print("\nALL OK" if all(results) else "\nSOMETHING IS WRONG, see the BAD lines above")

expand_deck comes from: a copy defined in this cell (tcg_bot/decks.py does not exist yet)

== lopunny

1) the .txt lines (line number = line_order)
 line_order section  avg_count                   name   set  number              card_name  matched_by
          1 Pokémon        1.0                Snorunt   ASC      46                Snorunt  set+number
          2 Pokémon        1.0              Dunsparce   TEF     128              Dunsparce  set+number
          3 Pokémon        3.0            Dudunsparce   TEF     129            Dudunsparce        name
          4 Pokémon        2.0                Buneary   PFL      83                Buneary  set+number
          5 Pokémon        2.0        Mega Lopunny ex   PFL     128        Mega Lopunny ex        name
          6 Pokémon        2.0              Dunsparce   JTG     120              Dunsparce  set+number
          7 Pokémon        1.0              Fan Rotom   SCR     118              Fan Rotom        name
          8 Pokémon        1

In [ ]:
# ---- print every card tag and what card it is ----
SHOW_DECK = "lopunny"           # change to "mega_lucario" for the other deck

names = cards.set_index("card_id")["card_name"]
for fam in ("pokemon", "trainer", "energy"):
    print(f"--- {fam} ---")
    for inst in expand_deck(deck_cards, cards, SHOW_DECK):
        if inst.tag.startswith(f"{SHOW_DECK}_{fam}_"):
            print(f"{inst.tag:<26} {names[inst.card_id]}  (card_id {inst.card_id})")

--- pokemon ---
lopunny_pokemon_1          Snorunt  (card_id 860)
lopunny_pokemon_2          Snorunt  (card_id 860)
lopunny_pokemon_3          Dunsparce  (card_id 65)
lopunny_pokemon_4          Dudunsparce  (card_id 66)
lopunny_pokemon_5          Dudunsparce  (card_id 66)
lopunny_pokemon_6          Dudunsparce  (card_id 66)
lopunny_pokemon_7          Buneary  (card_id 848)
lopunny_pokemon_8          Buneary  (card_id 848)
lopunny_pokemon_9          Mega Lopunny ex  (card_id 849)
lopunny_pokemon_10         Mega Lopunny ex  (card_id 849)
lopunny_pokemon_11         Dunsparce  (card_id 305)
lopunny_pokemon_12         Dunsparce  (card_id 305)
lopunny_pokemon_13         Fan Rotom  (card_id 174)
lopunny_pokemon_14         Chien-Pao  (card_id 209)
lopunny_pokemon_15         Psyduck  (card_id 858)
lopunny_pokemon_16         Mega Froslass ex  (card_id 861)
lopunny_pokemon_17         Mega Froslass ex  (card_id 861)
--- trainer ---
lopunny_trainer_1          Buddy-Buddy Poffin  (card_id 1086)
lopu

In [70]:
# ---- print every card tag and what card it is ----
SHOW_DECK = "mega_lucario"           # change to "mega_lucario" for the other deck

names = cards.set_index("card_id")["card_name"]
for fam in ("pokemon", "trainer", "energy"):
    print(f"--- {fam} ---")
    for inst in expand_deck(deck_cards, cards, SHOW_DECK):
        if inst.tag.startswith(f"{SHOW_DECK}_{fam}_"):
            print(f"{inst.tag:<26} {names[inst.card_id]}  (card_id {inst.card_id})")

--- pokemon ---
mega_lucario_pokemon_1     Riolu  (card_id 677)
mega_lucario_pokemon_2     Riolu  (card_id 677)
mega_lucario_pokemon_3     Riolu  (card_id 677)
mega_lucario_pokemon_4     Mega Lucario ex  (card_id 678)
mega_lucario_pokemon_5     Mega Lucario ex  (card_id 678)
mega_lucario_pokemon_6     Mega Lucario ex  (card_id 678)
mega_lucario_pokemon_7     Solrock  (card_id 676)
mega_lucario_pokemon_8     Solrock  (card_id 676)
mega_lucario_pokemon_9     Solrock  (card_id 676)
mega_lucario_pokemon_10    Makuhita  (card_id 673)
mega_lucario_pokemon_11    Makuhita  (card_id 673)
mega_lucario_pokemon_12    Hariyama  (card_id 674)
mega_lucario_pokemon_13    Hariyama  (card_id 674)
mega_lucario_pokemon_14    Lunatone  (card_id 675)
mega_lucario_pokemon_15    Lunatone  (card_id 675)
mega_lucario_pokemon_16    Meowth ex  (card_id 1071)
--- trainer ---
mega_lucario_trainer_1     Lillie's Determination  (card_id 1227)
mega_lucario_trainer_2     Lillie's Determination  (card_id 1227)
mega_luca